# External comparison — BhashaBench-Legal, the FULL set

Every number in this project so far comes from a benchmark we built ourselves; the first
external run (Aug 2026) used a 1,500-question sample and its CI was wide. This runs all
24,365 questions, letter-logit MCQ scoring (argmax over the four option letters' next-token logits; nothing generated, nothing unparsed), and keeps per-question rows so the result can be
sliced by language and subject afterwards.

**Settings:** GPU **T4 x2**, Internet **On**, Inputs: `jitendrajha98/nyaya-model-src` and `jitendrajha98/bhashabench-legal-cache` (the
24,365 questions as parquet; no Hugging Face token needed).
Two models per session on the same 3,000-question subset (`SUBSET`, seed 0), ~1.5 h; set `SUBSET = 0` for all 24k (~5-6 h per model).


In [ ]:
import os, subprocess, sys, json, re, time
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "transformers<5", "accelerate", "datasets"], check=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # T4 x2 -> DataParallel breaks placement
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

# Both readers in one session on the same deterministic subset, so the comparison is paired.
# 3,000 questions x 2 models is ~1.5 h on a T4; the full 24k x 2 would be ~11 h of a 30 h weekly quota.
MODELS = [("Qwen/Qwen3-4B-Instruct-2507", "qwen3-4b"), ("Qwen/Qwen2.5-3B-Instruct", "base")]

# A token is only needed when the questions come from the gated Hub dataset; the attached
# Kaggle dataset jitendrajha98/bhashabench-legal-cache holds the same 24,365 rows as parquet.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _label in ("HF_TOKEN", "Nyaya"):
        try:
            os.environ["HF_TOKEN"] = _secrets.get_secret(_label)
            break
        except Exception:
            continue
except Exception:
    pass

major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch})")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch build -- switch the accelerator to GPU T4 x2")
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("dtype:", DTYPE)


In [ ]:
import glob

from datasets import Dataset, concatenate_datasets, get_dataset_config_names, load_dataset

DATASET = "bharatgenai/BhashaBench-Legal"
parts = []
cached = sorted(glob.glob("/kaggle/input/**/bhashabench-legal-cache/**/*.parquet", recursive=True))
if cached:
    import pandas as pd
    for f in cached:
        lang = os.path.basename(f).rsplit(".", 1)[0]          # english / hindi
        df = pd.read_parquet(f)
        df["lang"] = lang
        parts.append(Dataset.from_pandas(df, preserve_index=False))
        print(f"{lang}: {len(df)} (Kaggle parquet copy)")
else:
    configs = get_dataset_config_names(DATASET, token=os.environ.get("HF_TOKEN"))
    print("configs:", configs)
    for cfg in configs:
        ds = load_dataset(DATASET, cfg, token=os.environ.get("HF_TOKEN"))
        split = "test" if "test" in ds else list(ds.keys())[0]
        part = ds[split].add_column("lang", [cfg] * len(ds[split]))
        parts.append(part)
        print(f"{cfg}/{split}: {len(part)}")
bench = concatenate_datasets(parts)
SUBSET = 3000                   # 0 = all questions; 3,000 keeps two models inside one session
if SUBSET:
    bench = bench.shuffle(seed=0).select(range(SUBSET))
print(len(bench), "questions |", bench.column_names)

for col in ("question", "option_a", "option_b", "correct_answer"):
    assert col in bench.column_names, f"missing column: {col}"
golds = {str(r).strip().upper()[:1] for r in bench["correct_answer"]}
assert golds <= set("ABCD"), f"unexpected gold labels: {golds - set('ABCD')}"
SUBJECT_COL = "subject_domain" if "subject_domain" in bench.column_names else next(
    (c for c in bench.column_names if c.lower() in ("subject", "topic", "domain", "subdomain", "category", "sub_domain")), None)
print("subject column:", SUBJECT_COL)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import gzip

LETTERS = ("A", "B", "C", "D")


def mcq_prompt(row):
    opts = chr(10).join(f"{L}. {row['option_' + L.lower()]}" for L in LETTERS
                        if row.get("option_" + L.lower()) not in (None, ""))
    return f"{row['question']}{chr(10)}{opts}{chr(10)}{chr(10)}Answer with the single letter of the correct option."


def letter_token_ids(tok):
    """First token id of each option letter, with and without a leading space. Scoring takes the
    max over the variants, so tokenizers that prefer ' B' and ones that prefer 'B' both work."""
    ids = {}
    for L in LETTERS:
        cands = set()
        for text in (L, " " + L):
            enc = tok.encode(text, add_special_tokens=False)
            if enc:
                cands.add(enc[0])
        ids[L] = sorted(cands)
    return ids


def run_mcq(model_id, label, batch_size=16):
    """Logit scoring: one forward pass per question, argmax over the four letters' next-token
    logits at the assistant position. Nothing is generated, so nothing can be 'unparsed' — the
    generation-based pass on 2026-09-04 left 522 of Qwen3-4B's 3,000 answers unparsed and was
    not comparable (reports/bhashabench_paired3000_generation.json)."""
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE, device_map={"": 0}).eval()
    ids = letter_token_ids(tok)
    print(f"[{label}] letter token ids: {ids}")
    correct = total = 0
    rows, t0 = [], time.time()
    for start in range(0, len(bench), batch_size):
        batch = bench.select(range(start, min(start + batch_size, len(bench))))
        texts = [tok.apply_chat_template([{"role": "user", "content": mcq_prompt(r)}],
                                         tokenize=False, add_generation_prompt=True) for r in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[:, -1, :].float()          # left padding: last position is live
        scores = torch.stack([logits[:, ids[L]].max(dim=1).values for L in LETTERS], dim=1)  # [B, 4]
        preds = [LETTERS[i] for i in scores.argmax(dim=1).tolist()]
        for row, pred in zip(batch, preds):
            gold = str(row["correct_answer"]).strip().upper()[:1]
            correct += pred == gold
            total += 1
            rows.append({"id": row.get("id"), "pred": pred, "gold": gold, "lang": row["lang"],
                         "subject": row.get(SUBJECT_COL) if SUBJECT_COL else None})
        if (start // batch_size) % 50 == 0:
            done, el = start + len(batch), time.time() - t0
            print(f"[{label}] {done}/{len(bench)} acc {correct/max(1,total):.1%} elapsed {el:.0f}s eta {el/done*(len(bench)-done):.0f}s", flush=True)
    by_lang, by_subj = {}, {}
    for r in rows:
        by_lang.setdefault(r["lang"], []).append(r["pred"] == r["gold"])
        if r["subject"] is not None:
            by_subj.setdefault(str(r["subject"]), []).append(r["pred"] == r["gold"])
    summary = {"model": model_id, "scoring": "letter-logit argmax", "accuracy": correct / total, "n": total,
               "unparsed": 0,
               "by_language": {k: sum(v) / len(v) for k, v in by_lang.items()},
               "by_subject": {k: {"n": len(v), "acc": sum(v) / len(v)} for k, v in sorted(by_subj.items())}}
    with gzip.open(f"/kaggle/working/bhashabench_full_{label}_rows.jsonl.gz", "wt", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + chr(10))
    json.dump(summary, open(f"/kaggle/working/bhashabench_full_{label}.json", "w"), indent=2)
    print(json.dumps({k: v for k, v in summary.items() if k != "by_subject"}, indent=2))
    del model
    torch.cuda.empty_cache()
    return summary


summaries = {label: run_mcq(model_id, label) for model_id, label in MODELS}


In [ ]:
# Standard error for one accuracy on n questions; the paired comparison against
# the other model is done on CPU afterwards from the saved rows.
import math
for label, summary in summaries.items():
    p, n = summary["accuracy"], summary["n"]
    se = math.sqrt(p * (1 - p) / n)
    print(f"{label}: {p:.1%} on {n} questions, 95% CI +/-{1.96*se:.2%}; chance is 25%  | by language {summary['by_language']}")
print("Download bhashabench_full_*.json and *_rows.jsonl.gz from the Output tab.")
